# 3.3 Sistemas de inferencia: hacia adelante y hacia atrás

**Asignatura:** Introducción a la Inteligencia Artificial  
**Unidad 3:** Representación del conocimiento y razonamiento

## Propósito

Implementar de forma didáctica dos mecanismos clásicos de inferencia:

a) **Encadenamiento hacia adelante** (*forward chaining*)

b) **Encadenamiento hacia atrás** (*backward chaining*)

El objetivo es observar cómo una base de hechos y una base de reglas permiten obtener nuevas conclusiones y, al mismo tiempo, registrar una **traza de inferencia**.


## 1. Base de conocimiento del ejemplo

Trabajaremos con un sistema sencillo de diagnóstico de un equipo.

### Hechos iniciales

- El equipo no enciende
- El LED está apagado
- La batería está descargada

### Reglas

**R1**  
SI el equipo no enciende Y el LED está apagado  
ENTONCES revisar la alimentación

**R2**  
SI se debe revisar la alimentación Y la batería está descargada  
ENTONCES posible falla de batería

**R3**  
SI existe una posible falla de batería  
ENTONCES recomendar revisión de batería


In [ ]:
hechos_iniciales = {
    "equipo_no_enciende",
    "led_apagado",
    "bateria_descargada"
}

reglas = [
    {
        "nombre": "R1",
        "condiciones": {
            "equipo_no_enciende",
            "led_apagado"
        },
        "conclusion": "revisar_alimentacion"
    },
    {
        "nombre": "R2",
        "condiciones": {
            "revisar_alimentacion",
            "bateria_descargada"
        },
        "conclusion": "posible_falla_bateria"
    },
    {
        "nombre": "R3",
        "condiciones": {
            "posible_falla_bateria"
        },
        "conclusion": "recomendar_revision_bateria"
    }
]

print("Hechos iniciales:")
for h in sorted(hechos_iniciales):
    print("-", h)


## 2. Forward chaining

El encadenamiento hacia adelante comienza con los hechos conocidos y busca reglas cuyas condiciones estén satisfechas.

La pregunta que guía el proceso es:

> **¿Qué puedo concluir con lo que sé?**


In [ ]:
def forward_chaining(hechos, reglas):
    hechos = set(hechos)
    traza = []
    cambio = True

    while cambio:
        cambio = False

        for regla in reglas:
            condiciones = regla["condiciones"]
            conclusion = regla["conclusion"]

            if condiciones.issubset(hechos) and conclusion not in hechos:
                hechos.add(conclusion)

                traza.append({
                    "regla": regla["nombre"],
                    "condiciones": sorted(condiciones),
                    "conclusion": conclusion
                })

                cambio = True

    return hechos, traza


In [ ]:
hechos_finales, traza_forward = forward_chaining(
    hechos_iniciales,
    reglas
)

print("Traza de forward chaining:")
for i, paso in enumerate(traza_forward, start=1):
    print(
        f"Paso {i}: {paso['regla']} -> {paso['conclusion']}"
    )

print("\nHechos finales:")
for h in sorted(hechos_finales):
    print("-", h)


### Interpretación

La ejecución esperada es:

```text
R1 → revisar_alimentacion
R2 → posible_falla_bateria
R3 → recomendar_revision_bateria
```

Esto muestra cómo una conclusión puede convertirse en un nuevo hecho y permitir la activación de otra regla.


## 3. Mostrar una traza más explicativa


In [ ]:
def mostrar_traza_forward(traza):
    if not traza:
        print("No se aplicó ninguna regla.")
        return

    for i, paso in enumerate(traza, start=1):
        print(f"\nPaso {i}")
        print("Regla:", paso["regla"])
        print("Condiciones:")
        for c in paso["condiciones"]:
            print("  ✓", c)
        print("Nuevo hecho:")
        print("  ->", paso["conclusion"])

mostrar_traza_forward(traza_forward)


## 4. Backward chaining

El encadenamiento hacia atrás comienza con una meta y busca qué reglas podrían demostrarla.

La pregunta que guía el proceso es:

> **¿Qué necesito demostrar para confirmar esta meta?**


In [ ]:
def backward_chaining(meta, hechos, reglas, nivel=0, visitados=None):
    if visitados is None:
        visitados = set()

    indentacion = "  " * nivel
    print(f"{indentacion}Meta: {meta}")

    if meta in hechos:
        print(f"{indentacion}✓ Es un hecho conocido")
        return True

    if meta in visitados:
        print(f"{indentacion}✗ Meta ya visitada; se evita un ciclo")
        return False

    visitados.add(meta)

    reglas_candidatas = [
        regla for regla in reglas
        if regla["conclusion"] == meta
    ]

    if not reglas_candidatas:
        print(f"{indentacion}✗ No existe un hecho ni una regla que demuestre la meta")
        return False

    for regla in reglas_candidatas:
        print(f"{indentacion}Usando {regla['nombre']}")

        resultados = []

        for condicion in sorted(regla["condiciones"]):
            resultado = backward_chaining(
                condicion,
                hechos,
                reglas,
                nivel + 1,
                visitados.copy()
            )
            resultados.append(resultado)

        if all(resultados):
            print(f"{indentacion}✓ Meta demostrada: {meta}")
            return True

    print(f"{indentacion}✗ No se pudo demostrar: {meta}")
    return False


In [ ]:
meta = "recomendar_revision_bateria"

resultado = backward_chaining(
    meta,
    hechos_iniciales,
    reglas
)

print("\nResultado final:", resultado)


## 5. Comparación de ambos mecanismos

Con la misma base de conocimiento:

### Forward chaining

Parte de:

```text
equipo_no_enciende
led_apagado
bateria_descargada
```

y genera conclusiones.

### Backward chaining

Parte de:

```text
recomendar_revision_bateria
```

y busca qué condiciones deben demostrarse.

| Aspecto | Forward chaining | Backward chaining |
|---|---|---|
| Inicio | Hechos | Meta |
| Dirección | Datos → conclusión | Meta → datos |
| Pregunta | ¿Qué puedo concluir? | ¿Puedo demostrar esto? |
| Estrategia | Dirigida por datos | Dirigida por metas |


## 6. Experimento 1: eliminar un hecho

Elimine temporalmente el hecho:

```text
bateria_descargada
```

y observe qué ocurre con ambos mecanismos.


In [ ]:
hechos_sin_bateria = {
    "equipo_no_enciende",
    "led_apagado"
}

hechos_finales_2, traza_forward_2 = forward_chaining(
    hechos_sin_bateria,
    reglas
)

print("Forward chaining:")
mostrar_traza_forward(traza_forward_2)

print("\nBackward chaining:")
resultado_2 = backward_chaining(
    "recomendar_revision_bateria",
    hechos_sin_bateria,
    reglas
)

print("\nResultado:", resultado_2)


### Preguntas de análisis

a) ¿Qué regla sigue siendo aplicable?

b) ¿Qué conclusión deja de obtenerse?

c) ¿El hecho `bateria_descargada` es falso o simplemente desconocido?

d) ¿Qué diferencia observa entre el resultado de forward y backward chaining?


## 7. Experimento 2: agregar una nueva regla

Agregue una regla alternativa para explicar por qué un equipo no enciende.

Ejemplo:

```text
R4:
SI equipo_no_enciende
Y fuente_danada
ENTONCES posible_falla_fuente
```


In [ ]:
reglas_extendidas = reglas + [
    {
        "nombre": "R4",
        "condiciones": {
            "equipo_no_enciende",
            "fuente_danada"
        },
        "conclusion": "posible_falla_fuente"
    }
]

hechos_extendidos = set(hechos_iniciales)
hechos_extendidos.add("fuente_danada")

hechos_finales_3, traza_forward_3 = forward_chaining(
    hechos_extendidos,
    reglas_extendidas
)

mostrar_traza_forward(traza_forward_3)


### Preguntas de análisis

a) ¿Cuántas conclusiones diferentes pueden obtenerse ahora?

b) ¿Puede un mismo síntoma tener más de una explicación?

c) ¿Qué problema aparecería si muchas reglas pudieran activarse al mismo tiempo?


## 8. Ejemplo adicional: sistema de riego

Aplique forward chaining a la siguiente base de conocimiento.


In [ ]:
hechos_riego = {
    "suelo_seco",
    "no_llueve",
    "deposito_con_agua"
}

reglas_riego = [
    {
        "nombre": "R1",
        "condiciones": {
            "suelo_seco",
            "no_llueve"
        },
        "conclusion": "activar_riego"
    },
    {
        "nombre": "R2",
        "condiciones": {
            "activar_riego",
            "deposito_con_agua"
        },
        "conclusion": "abrir_valvula"
    },
    {
        "nombre": "R3",
        "condiciones": {
            "abrir_valvula"
        },
        "conclusion": "regando"
    }
]

hechos_riego_finales, traza_riego = forward_chaining(
    hechos_riego,
    reglas_riego
)

mostrar_traza_forward(traza_riego)


## 9. Actividad breve

Modifique la base de conocimiento del sistema de riego de manera que incluya al menos:

a) Un nuevo hecho

b) Una nueva regla

c) Una nueva conclusión

Después:

a) Ejecute forward chaining

b) Plantee una meta y ejecútela mediante backward chaining

c) Explique la traza obtenida

d) Indique qué conocimiento fue inicial y cuál fue derivado


## 10. Conclusión

Este notebook muestra que:

a) **Forward chaining** parte de los hechos y deriva nuevas conclusiones

b) **Backward chaining** parte de una meta e intenta demostrarla

c) Las reglas conectan conocimiento conocido con conocimiento derivado

d) Las trazas permiten explicar el razonamiento del sistema

e) La ausencia de un hecho no implica necesariamente falsedad

La implementación utilizada es deliberadamente sencilla y tiene un propósito didáctico. No representa todas las características de un motor de inferencia real.
